# การวิเคราะห์การเคลมค่าใช้จ่าย

โน้ตบุ๊กนี้สาธิตวิธีการสร้างเอเย่นต์ที่ใช้ปลั๊กอินในการประมวลผลค่าใช้จ่ายเดินทางจากภาพใบเสร็จรับเงินท้องถิ่น สร้างอีเมลเคลมค่าใช้จ่าย และแสดงข้อมูลค่าใช้จ่ายด้วยแผนภูมิวงกลม เอเย่นต์จะเลือกรันฟังก์ชันตามบริบทของงานอย่างไดนามิก

ขั้นตอน:
1. เอเย่นต์ OCR ประมวลผลภาพใบเสร็จรับเงินท้องถิ่นและดึงข้อมูลค่าใช้จ่ายเดินทางออกมา
2. เอเย่นต์อีเมลสร้างอีเมลเคลมค่าใช้จ่าย

### ตัวอย่างสถานการณ์ค่าใช้จ่ายเดินทาง:
สมมติว่าคุณเป็นพนักงานที่เดินทางเพื่อประชุมธุรกิจในเมืองอื่น บริษัทของคุณมีนโยบายคืนเงินค่าใช้จ่ายที่เกี่ยวข้องกับการเดินทางที่สมเหตุสมผลทั้งหมด นี่คือการแบ่งค่าใช้จ่ายเดินทางที่เป็นไปได้:
- การเดินทาง:
ค่าตั๋วเครื่องบินไปกลับจากเมืองบ้านเกิดของคุณไปยังเมืองปลายทาง
ค่ารถแท็กซี่หรือบริการเรียกรถไปและกลับสนามบิน
การเดินทางภายในเมืองปลายทาง (เช่น รถสาธารณะ รถเช่า หรือแท็กซี่)

- ที่พัก:
การเข้าพักโรงแรมสามคืนที่โรงแรมธุรกิจระดับกลางใกล้สถานที่ประชุม

- ค่าอาหาร:
ค่าเบี้ยเลี้ยงรายวันสำหรับอาหารเช้า กลางวัน และเย็น ตามนโยบายรายวันของบริษัท

- ค่าใช้จ่ายเบ็ดเตล็ด:
ค่าจอดรถที่สนามบิน
ค่าใช้บริการอินเทอร์เน็ตที่โรงแรม
ทิปหรือค่าบริการเล็กน้อย

- เอกสารประกอบ:
คุณส่งใบเสร็จทั้งหมด (ตั๋วเครื่องบิน แท็กซี่ โรงแรม อาหาร ฯลฯ) และรายงานค่าใช้จ่ายที่กรอกครบถ้วนเพื่อขอคืนเงิน


## Import required libraries

นำเข้าห้องสมุดและโมดูลที่จำเป็นสำหรับโน้ตบุ๊กนี้


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Define Expense Models

สร้างโมเดล Pydantic สำหรับรายจ่ายแต่ละรายการและคลาส ExpenseFormatter เพื่อแปลงคำค้นหาของผู้ใช้เป็นข้อมูลรายจ่ายที่มีโครงสร้าง

รายจ่ายแต่ละรายการจะแสดงในรูปแบบ:
`{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Defining Tools - การสร้างอีเมล

สร้างฟังก์ชันเครื่องมือเพื่อสร้างอีเมลสำหรับส่งคำร้องขอค่าใช้จ่าย  
- เครื่องมือนี้ใช้ตัวตกแต่ง `@tool` จาก Microsoft Agent Framework  
- คำนวณจำนวนรวมของค่าใช้จ่ายและจัดรูปแบบรายละเอียดเป็นเนื้อหาอีเมล


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# เครื่องมือสำหรับดึงข้อมูลค่าใช้จ่ายการเดินทางจากรูปภาพใบเสร็จ

สร้างฟังก์ชันเครื่องมือเพื่อดึงข้อมูลค่าใช้จ่ายการเดินทางจากรูปภาพใบเสร็จ
- เครื่องมือนี้ใช้ตัวตกแต่ง `@tool` จาก Microsoft Agent Framework
- มันอ่านรูปภาพใบเสร็จ, เข้ารหัสเป็น base64 และส่งคืน data URI เพื่อให้ตัวแทนวิเคราะห์ต่อไป


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## การประมวลผลค่าใช้จ่าย

กำหนดเอเย่นต์และเชื่อมต่อพวกเขาเข้ากับเวิร์กโฟลว์แบบลำดับโดยใช้ `WorkflowBuilder`
- เอเย่นต์ OCR ดึงข้อมูลค่าใช้จ่ายแบบมีโครงสร้างจากภาพใบเสร็จโดยใช้เครื่องมือ `load_receipt_image`
- เอเย่นต์อีเมลรับข้อมูลที่ดึงออกมาและสร้างอีเมลเรียกร้องค่าใช้จ่ายอย่างมืออาชีพโดยใช้เครื่องมือ `generate_expense_email`
- `WorkflowBuilder` พร้อม `add_edge` สร้างสายงานแบบลำดับ: เอเย่นต์ OCR → เอเย่นต์อีเมล


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## ฟังก์ชันหลัก

สร้างเวิร์กโฟลว์แบบลำดับขั้นและเรียกใช้งานเพื่อประมวลผลภาพใบเสร็จและสร้างอีเมลขอคืนเงินค่าใช้จ่าย


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ข้อจำกัดความรับผิดชอบ**:
เอกสารฉบับนี้ได้รับการแปลโดยใช้บริการแปลภาษาด้วย AI [Co-op Translator](https://github.com/Azure/co-op-translator) แม้เราจะพยายามให้มีความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่แม่นยำ เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดใด ๆ ที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
